# M2 Hadoop WordCount Practice — PC / VS Code Notebook

This notebook is designed to run **line by line in VS Code** on a Windows PC.

It does **not** use Google Colab code such as:

```python
from google.colab import drive
!apt-get install ...
%%bash
```

## Goal

You will:

1. Set Java and Hadoop paths.
2. Create a small text file named `student_text.txt`.
3. Run Hadoop WordCount only on that file.
4. View the WordCount output.
5. Practice by editing the file and rerunning WordCount.


## Step 1 — Import libraries

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

## Step 2 — Set Java and Hadoop paths

Update these two paths if your PC uses different installation folders.


In [ ]:
JAVA_HOME = r"C:\Program Files\Java\jdk-11"
HADOOP_HOME = r"C:\hadoop-3.4.1"
HADOOP_VERSION = "3.4.1"

os.environ["JAVA_HOME"] = JAVA_HOME
os.environ["HADOOP_HOME"] = HADOOP_HOME
os.environ["PATH"] = (
    str(Path(HADOOP_HOME) / "bin")
    + os.pathsep
    + str(Path(HADOOP_HOME) / "sbin")
    + os.pathsep
    + os.environ.get("PATH", "")
)

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])

## Step 3 — Define working folders and Hadoop command paths

In [ ]:
USER_HOME = Path.home()

INPUT_DIR = USER_HOME / "student_wordcount_input"
OUTPUT_DIR = USER_HOME / "student_wordcount_output"
STUDENT_FILE = INPUT_DIR / "student_text.txt"

HADOOP_CMD = Path(HADOOP_HOME) / "bin" / "hadoop.cmd"

MAPREDUCE_EXAMPLES_JAR = (
    Path(HADOOP_HOME)
    / "share"
    / "hadoop"
    / "mapreduce"
    / f"hadoop-mapreduce-examples-{HADOOP_VERSION}.jar"
)

print("Input folder:", INPUT_DIR)
print("Output folder:", OUTPUT_DIR)
print("Student file:", STUDENT_FILE)
print("Hadoop command:", HADOOP_CMD)
print("MapReduce examples jar:", MAPREDUCE_EXAMPLES_JAR)

## Step 4 — Check installation

Run this cell to confirm Java and Hadoop are available.


In [ ]:
print("Checking paths...")

if not Path(JAVA_HOME).exists():
    print("WARNING: JAVA_HOME does not exist. Please update JAVA_HOME.")

if not Path(HADOOP_HOME).exists():
    raise FileNotFoundError(f"HADOOP_HOME does not exist: {HADOOP_HOME}")

if not HADOOP_CMD.exists():
    raise FileNotFoundError(f"Hadoop command not found: {HADOOP_CMD}")

if not MAPREDUCE_EXAMPLES_JAR.exists():
    raise FileNotFoundError(f"MapReduce examples jar not found: {MAPREDUCE_EXAMPLES_JAR}")

print("Basic path checks passed.")

## Step 5 — Helper function to run terminal commands

This lets the notebook run Hadoop commands from Python.


In [ ]:
def run_command(command, allow_fail=False):
    print("-" * 60)
    print("Running command:")
    print(" ".join(map(str, command)))
    print("-" * 60)

    result = subprocess.run(
        command,
        text=True,
        capture_output=True,
        env=os.environ.copy()
    )

    if result.stdout:
        print("STDOUT:")
        print(result.stdout)

    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed with return code {result.returncode}")

    return result

## Step 6 — Verify Java and Hadoop

In [ ]:
run_command(["java", "-version"])
run_command([str(HADOOP_CMD), "version"])

## Step 7 — Create a clean input folder and student text file

This cell creates **one small input file** only.

This avoids accidentally reading a large folder.


In [ ]:
# Clean old input and output folders
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

INPUT_DIR.mkdir(parents=True, exist_ok=True)

student_text = """Hadoop helps count words
Hadoop works with big data
Students practice Hadoop WordCount
Data data data is everywhere
WordCount counts each word
"""

STUDENT_FILE.write_text(student_text, encoding="utf-8")

print("Student file created:")
print(STUDENT_FILE)

print("\\nStudent file content:")
print(STUDENT_FILE.read_text(encoding="utf-8"))

## Step 8 — Run Hadoop WordCount on only `student_text.txt`

Important: this reads only:

```text
student_text.txt
```

It does **not** read the whole input folder.


In [ ]:
# Remove old WordCount output folder before running Hadoop
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

wordcount_command = [
    str(HADOOP_CMD),
    "jar",
    str(MAPREDUCE_EXAMPLES_JAR),
    "wordcount",
    str(STUDENT_FILE),
    str(OUTPUT_DIR)
]

run_command(wordcount_command)

## Step 9 — Display WordCount output

In [ ]:
output_files = sorted(OUTPUT_DIR.glob("part-*"))

if not output_files:
    print("No output file found.")
else:
    for output_file in output_files:
        print("Output file:", output_file)
        print(output_file.read_text(encoding="utf-8", errors="ignore"))

# Student Practice

Now students should edit the input file and rerun WordCount.

## Practice instructions

1. Open this file on your PC:

```text
student_wordcount_input/student_text.txt
```

2. Replace the text with your own 5 or more lines.

Example idea:

```text
People in those industries often travel globally.
They move between projects.
They work with contractors and business groups.
Some periods are very intense.
Other periods are quieter.
```

3. Rerun:

- Step 8
- Step 9

4. Answer:

- Which word appears most often?
- Did the output change?
- Why did the count change?
- What is the difference between running WordCount on one file versus the entire folder?


## Optional Practice Cell — Update `student_text.txt` from inside the notebook

Students may use this cell instead of manually opening the file.

They can change the text below, then rerun Step 8 and Step 9.


In [ ]:
new_student_text = """People in those industries often travel globally
They move between projects
They work with multiple contractors and business groups
Some periods are very intense
Other periods are quieter
"""

STUDENT_FILE.write_text(new_student_text, encoding="utf-8")

print("Updated student_text.txt:")
print(STUDENT_FILE.read_text(encoding="utf-8"))